# Incremental Data Ingestion & Data Contracts

## Objective

In Notebook 08, I introduced Airflow to coordinate the existing Snowflake/dbt warehouse refresh. That workflow rebuilds analytical tables from the same fixed historical X5 snapshot.

In this notebook, I explore a different problem: processing newly arriving data without rebuilding the entire historical dataset.

Because X5 does not provide a live event feed, I use a small, explicitly synthetic dataset to demonstrate incremental ingestion.

The workflow generates two batches of purchase-like events, stores them in S3, loads them into an isolated Snowflake landing table, validates their contents, quarantines invalid records, and promotes valid unique events into a trusted table.

I focus on four engineering properties: incremental processing, idempotency, data contracts, and source-level traceability.

The demonstration does not modify the original X5 RAW tables, dbt marts, machine-learning features, or deployed uplift model.

## 1. Incremental ingestion architecture

The incremental workflow separates raw data preservation from downstream validation.

The raw landing table stores source values as strings, together with the originating filename, source row number, and load timestamp. This allows invalid records to be investigated without losing their original contents.

A validation layer checks required identifiers, schema version, timestamp parsing, amount parsing, and conflicting event IDs.

Valid records are promoted into a trusted event table using a deduplicated `MERGE`. Invalid records are preserved in a separate quarantine table with an explicit rejection reason.

This design provides two levels of duplicate protection:

1. Snowflake's file-loading history helps avoid reloading previously processed staged files.
2. The business-key `MERGE` prevents previously accepted event IDs from being inserted again.

File-level protection alone is insufficient because the same business event can appear in multiple source files.

In [1]:
# ============================================================
# Generate the synthetic source batches
#
# The generator is shared with the command-line workflow.
# The notebook does not maintain a second copy of the data.
# ============================================================

from pathlib import Path
import csv
import sys

import pandas as pd


PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dbt" / "dbt_project.yml").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT),
)

from scripts.generate_incremental_demo import (
    generate_demo_files,
    validate_csv_header,
)


OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "incremental_demo"
)


paths = generate_demo_files(
    OUTPUT_DIR
)


# ------------------------------------------------------------
# Inspect the input records and verify their CSV contracts.
# ------------------------------------------------------------

batch_summary = []

for path in paths:

    validate_csv_header(path)

    batch = pd.read_csv(path)

    batch_summary.append(
        {
            "file": path.name,
            "rows": len(batch),
            "columns": len(batch.columns),
        }
    )


display(
    pd.DataFrame(batch_summary)
)

,file,rows,columns
0,batch_01.csv,4,5
1,batch_02.csv,4,5


## 2. Data validation and quarantine

I intentionally include two invalid records in the second synthetic batch: one with a nonnumeric amount and one with an invalid timestamp.

Rather than silently dropping those records, I preserve them in the landing table and classify their rejection reasons.

The quarantine table provides an auditable record of what was rejected, where it came from, and why it could not be promoted.

The second batch also contains an exact duplicate of an event from the first batch. This tests whether the trusted event table maintains one row per business event even when the same event appears in multiple files.

The demonstration uses an append-only event model. Corrections to previously accepted events would require a separate reconciliation policy.

## 3. Incremental ingestion results

### First batch

- Source files processed: **4**
- Raw rows: **4**
- Trusted events: **4**
- Quarantined records: **0**

### Second batch

- Total source files processed: **4**
- Total raw rows: **8**
- Trusted unique events: **5**
- Quarantined records: **2**
- Repeated event IDs in the raw layer: **1**

### Idempotency check

I reran the loading and transformation commands without introducing additional source files.

- Trusted event count before rerun: **5**
- Trusted event count after rerun: **5**
- Quarantine count before rerun: **2**
- Quarantine count after rerun: **2**

The trusted and quarantine populations should remain unchanged when the same source data is processed again.


## 4. Conclusions and limitations

I built an isolated incremental-ingestion demonstration using synthetic event batches, the existing S3 infrastructure, and Snowflake.

The workflow preserves incoming source values and file-level metadata, validates record contents, quarantines invalid records, and promotes valid unique events into a trusted table.

I tested the behavior across two batches containing new events, an exact duplicate, an invalid amount, and an invalid timestamp. I also reran the ingestion and transformation logic to examine whether repeated execution changes the final trusted population.

### Key findings

- Source batches processed: **2**
- Raw records loaded: **8**
- Trusted unique events: **5**
- Quarantined records: **2**
- Duplicate business events handled: **1**
- Idempotency verification: **Success**
- Source-contract unit tests: **3**
- Full Python test suite: **14**

### Limitations

This demonstration uses explicitly synthetic purchase-like events rather than newly observed X5 transactions. Its outputs must not be combined with the original X5 customer marts, business metrics, or uplift-model training data.

The workflow is manually executed and does not yet detect new S3 objects automatically.

The implementation assumes append-only source files and a single writer to the trusted table. It does not implement an operational correction policy for previously accepted events or guarantee exactly-once processing under concurrent writers.

The CSV header contract is checked before upload. Additional controls would be required to verify producer identity, enforce schemas at the ingestion boundary, and handle unexpected changes to files after validation.

The next stage is to integrate incremental processing with orchestration and establish stronger operational reliability and monitoring controls.